In [ ]:
import os

import imageio.v3 as imageio
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("gif_frames", exist_ok=True)

# Parameters for continuous Brownian motion.
num_frames = 100
t_values = np.linspace(0.1, 5, num_frames)
x_values = np.linspace(-5, 5, 200)
num_sample_paths = 5
sample_paths = np.zeros((num_sample_paths, len(t_values)))
dt_cont = t_values[1] - t_values[0]

for path_index in range(num_sample_paths):
    increments = np.random.normal(
        0, np.sqrt(dt_cont), size=len(t_values) - 1
    )
    sample_paths[path_index, 1:] = np.cumsum(increments)

frames = []
for frame_index, time_value in enumerate(t_values):
    figure = plt.figure(figsize=(10, 7))
    axes = figure.add_subplot(111, projection="3d")

    mask = t_values <= time_value
    time_grid, position_grid = np.meshgrid(t_values[mask], x_values)
    density_grid = (1 / np.sqrt(2 * np.pi * time_grid)) * np.exp(
        -(position_grid**2) / (2 * time_grid)
    )

    axes.plot_surface(
        position_grid,
        time_grid,
        density_grid,
        cmap="viridis",
        alpha=0.7,
        edgecolor="none",
    )

    for sample_path in sample_paths:
        axes.plot(
            sample_path[: frame_index + 1],
            t_values[: frame_index + 1],
            np.zeros(frame_index + 1),
            "r-",
            marker="o",
            markersize=3,
        )

    axes.set_xlabel("Position (x)")
    axes.set_ylabel("Time (t)")
    axes.set_zlabel("Density")
    axes.set_title(f"Continuous Brownian Motion at t = {time_value:.2f}")
    axes.view_init(elev=30, azim=-60)

    frame_path = os.path.join(
        "gif_frames", f"continuous_3d_smooth_t_{time_value:.2f}.png"
    )
    figure.savefig(frame_path)
    plt.close(figure)
    frames.append(imageio.imread(frame_path))

imageio.imwrite(
    "continuous_brownian_3d_smooth.gif",
    frames,
    duration=0.1,
    loop=0,
)